In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [33]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()
#On remplace les valeurs manquantes de P1_reel, V et Lq_reel par la première valeur non nulle de chaque essai selon les conseils de l'encadrante
#Cela permet de conserver d'avantage de lignes avant le dropna()
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])

df = df[df['Lt'] == 1]



In [34]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','Lq_reel','P4','E2','E1','E2.1','E2.2','E2.3','E2.4','S_moy','sigma_S']]
df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
df_ut['essai'] = df['essai']

X1 = df_ut.drop(columns=['S_moy','sigma_S'])  # Features
X1['E2_moy'] = X1[['E2.1','E2.2','E2.3','E2.4']].mean(axis=1)  # Moyenne des colonnes E2.1, E2.2, E2.3, E2.4
X1.drop(columns=['E2.1','E2.2','E2.3','E2.4'], inplace=True)  # Supprimer les colonnes E2.1, E2.2, E2.3, E2.4
y1 = df_ut[['essai','S_moy']]  # Target

# 4. Split par groupe
groupes = X1['essai']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X1, y1, groups=groupes))

X_train, X_test = X1.iloc[train_idx].copy(), X1.iloc[test_idx].copy()
y_train, y_test = y1.iloc[train_idx].copy(), y1.iloc[test_idx].copy()
groupes_train = groupes.iloc[train_idx]

# 5. On retire l'essai pour ne pas tricher
X_train = X_train.drop(columns=['essai'])
X_test = X_test.drop(columns=['essai'])
y_test = y_test['S_moy']
y_train = y_train['S_moy']
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(707, 6) (144, 6) (707,) (144,)


In [ ]:
# Recherche des meilleurs hyperparamètres pour le Random Forest sur S_moy
param_grid = {
    'n_estimators': [100, 200, 600, 800],      # On cherche le bon nombre d'arbres
    'max_depth': [5, 8, 12, None],             # On cherche la bonne profondeur (PAS DE 'None' !)
    'min_samples_leaf': [2, 3, 5]        # On cherche à lisser plus ou moins les feuilles
}

# Modèle de base pour la recherche
rf_base = RandomForestRegressor(random_state=42)

print("Recherche des meilleurs paramètres pour S_moy en cours...")
# cv=5 : Validation croisée en 5 blocs. n_jobs=-1 : Utilise tout le CPU.
grid_moy = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_moy.fit(X_train, y_train)
rf_moy = grid_moy.best_estimator_
print(f"Meilleurs paramètres S_moy : {grid_moy.best_params_}")

print("--- 4. Évaluation ---")
# Prédictions (avec les meilleurs modèles trouvés automatiquement)
pred_moy = rf_moy.predict(X_test)

# Métriques pour S_moy
print(">> PERFORMANCES SUR S_moy (La valeur moyenne) :")
print(f"MAE  : {mean_absolute_error(y_test, pred_moy):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_moy)):.4f}")
print(f"R2   : {r2_score(y_test, pred_moy):.4f}\n")

Recherche des meilleurs paramètres pour S_moy en cours...
Meilleurs paramètres S_moy : {'max_depth': 12, 'min_samples_leaf': 2, 'n_estimators': 200}
--- 4. Évaluation ---
>> PERFORMANCES SUR S_moy (La valeur moyenne) :
MAE  : 0.2193
RMSE : 0.2349
R2   : 0.3882



In [35]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= 8,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")


R2: 0.379548
MSE: 0.055953

Erreur Absolue Moyenne (MAE) : 0.22
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.24 


--- 4. Importance des Paramètres ---
P4        : 61.5%
Lq_reel   : 24.7%
P1_reel   : 12.9%
E1        : 0.8%
essai     : 0.0%
E2        : 0.0%
